# What Is It?

Inspect a Terraform log file and determine what it contains:

- **export** — Genesys Cloud resource exporter activity
- **plan** — plan activity (`terraform plan -json` UI output or `TF_LOG=json` trace)
- **apply** — apply activity (`terraform apply -json` UI output or `TF_LOG=json` trace)
- **all of the above** — more than one category is present
- **none of the above** — no recognizable export/plan/apply markers

Set `TERRAFORM_LOG_PATH` before running (see repo `README.md`).

In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "commonlib" / "config.py").is_file():
        _notebooks_root = _root
        break
    if (_root / "notebooks" / "commonlib" / "config.py").is_file():
        _notebooks_root = _root / "notebooks"
        break
else:
    raise RuntimeError(
        "Could not find notebooks/commonlib/. Start Jupyter from notebooks/ "
        "or open a notebook under export/, plan/, apply/, sdk-plan/, or the notebooks root."
    )

if str(_notebooks_root) not in sys.path:
    sys.path.insert(0, str(_notebooks_root))

from commonlib.notebook_setup import setup

setup()

import pandas as pd

import commonlib.classify_tf_log as classify_tf_log
import commonlib.config as cfg


In [ ]:
c = cfg.Config()
log_path = c.TERRAFORM_LOG_PATH

if not log_path:
    raise ValueError(
        "Set TERRAFORM_LOG_PATH to your log file. "
        "Example: export TERRAFORM_LOG_PATH=/path/to/plan-tflog.log"
    )

print(f"Analyzing: {log_path}")
result = classify_tf_log.classify_file(log_path)
summary = classify_tf_log.classification_summary(result)

## Verdict

In [ ]:
print(f"Verdict: {summary['verdict']}")
print(f"Categories detected: {summary['categories'] or ['(none)']}")
print(f"Primary log format: {summary['primary_log_format']}")
print()
print(f"Lines: {summary['parsed_lines']:,} parsed / {summary['total_lines']:,} total")
if summary['parse_errors']:
    print(f"Parse errors: {summary['parse_errors']:,}")
if summary['terraform_version']:
    print(f"Terraform version: {summary['terraform_version']}")
if summary['first_timestamp']:
    print(f"Time range: {summary['first_timestamp']} → {summary['last_timestamp']}")

print("\nEvidence:")
if summary['evidence']:
    for line in summary['evidence']:
        print(f"  • {line}")
else:
    print("  • (no export/plan/apply markers found)")

if summary['warnings']:
    print("\nNotes:")
    for line in summary['warnings']:
        print(f"  • {line}")

## Which analysis notebook to use?

In [ ]:
recommendations = []

if summary['is_export']:
    recommendations.append("export/performance-analysis.ipynb — export performance analysis (completed run) and resource breakdown")
    recommendations.append("export/hang-analysis.ipynb — hung/slow export (stuck resources, SDK retries, 404 loops)")
if summary['is_plan']:
    if summary['log_formats'].get('terraform.ui'):
        recommendations.append("sdk-plan/plan-analysis.ipynb — plan-analysis (terraform plan -json UI output)")
    else:
        recommendations.append("plan/performance-analysis.ipynb — plan performance analysis (completed TF_LOG)")
        recommendations.append("plan/hang-analysis.ipynb — hung/slow plan (graph waits, SDK retries, 404 loops)")
if summary['is_apply']:
    recommendations.append("apply/performance-analysis.ipynb — apply performance analysis (completed run) and resource breakdown")
    recommendations.append("apply/hang-analysis.ipynb — hung/slow apply (stuck resources, open RPCs, SDK retries)")

if summary['is_export'] or summary['is_plan'] or summary['is_apply']:
    recommendations.append("sdk-plan/sdk-analysis.ipynb — optional SDK charts and raw pairs (TF_LOG captures)")
    recommendations.append("explore-capture.ipynb — load sidecars and tinker (filters, custom charts, commonlib)")

if not recommendations:
    recommendations.append(
        "No matching analysis notebook — check that the file is TF_LOG=json or terraform UI JSON output."
    )

for rec in recommendations:
    print(f"• {rec}")

## Breakdown

In [ ]:
overview = pd.DataFrame([
    {"metric": "export", "detected": summary["is_export"],
     "detail": f"starts={result.export_start_count}, ends={result.export_end_count}, exporter_lines={result.exporter_caller_count}"},
    {"metric": "plan", "detected": summary["is_plan"],
     "detail": f"ui_events={result.plan_ui_count}, change_summary(plan)={result.plan_change_summary_count}, plan_rpc={result.plan_rpc_count}"},
    {"metric": "apply", "detected": summary["is_apply"],
     "detail": f"ui_events={result.apply_ui_count}, change_summary(apply)={result.apply_change_summary_count}, apply_rpc={result.apply_rpc_count}"},
])
overview

In [ ]:
def counter_to_df(name, items):
    if not items:
        return pd.DataFrame(columns=[name, "count"])
    return pd.DataFrame(items, columns=[name, "count"])

display(counter_to_df("log_format", list(summary["log_formats"].items())))
display(counter_to_df("ui_type", summary["top_ui_types"]))
display(counter_to_df("tf_rpc", summary["top_tf_rpc"]))
display(counter_to_df("module", summary["top_modules"]))